# Training

In [ ]:
!pip install -q ml-collections

In [ ]:
import tensorflow as tf
# Set the device to CPU
tf.config.set_visible_devices([], 'GPU')

In [ ]:
import os
import urllib.request
from urllib.error import HTTPError
import ml_collections
import jax
from jax import numpy as jnp

main_rng_key = jax.random.key(18)

In [ ]:
!rm -rf de_tokenizer_20_000_vocab_size_model
!rm -rf en_tokenizer_20_000_vocab_size_model
!rm -rf log_dir
!rm -f configs.py tokenizer.py data.py model.py training_utils.py


!mkdir de_tokenizer_20_000_vocab_size_model
!mkdir en_tokenizer_20_000_vocab_size_model

In [ ]:
base_url = "https://raw.githubusercontent.com/MiguelSteph/transformer-from-scratch/main/"
def download_file_from_github(file_path: str, file_name: str):
    if not os.path.isfile(file_name):
        file_url = base_url + file_path
        print(f"Downloading {file_url}...")
        try:
            urllib.request.urlretrieve(file_url, file_name)
        except HTTPError as e:
            print("Something went wrong. Please try to download the file directly from the GitHub repository:\n", e)

file_paths = [
    'configs/configs.py',
    'data/tokenizer.py',
    'data/data.py',
    'models/model.py',
    'training/training_utils.py',
    'data/en_tokenizer_20_000_vocab_size_model/merges.txt',
    'data/en_tokenizer_20_000_vocab_size_model/vocab.json',
    'data/de_tokenizer_20_000_vocab_size_model/merges.txt',
    'data/de_tokenizer_20_000_vocab_size_model/vocab.json',
]
file_names = [
    'configs.py',
    'tokenizer.py',
    'data.py',
    'model.py',
    'training_utils.py',
    'en_tokenizer_20_000_vocab_size_model/merges.txt',
    'en_tokenizer_20_000_vocab_size_model/vocab.json',
    'de_tokenizer_20_000_vocab_size_model/merges.txt',
    'de_tokenizer_20_000_vocab_size_model/vocab.json',
]

for file_path, file_name in zip(file_paths, file_names): 
    download_file_from_github(file_path, file_name)

In [ ]:
from configs import get_configs
from model import create_transformer_module
from training_utils import train_and_evaluate, get_dataset_iterator, create_train_state, generate_random_batch, train_step
from data import load_preprocessed_dataset

base_configs = get_configs()
config = ml_collections.ConfigDict(base_configs)
config.data.test_ds_path = '/kaggle/input/datasets/migsena/de-en-preprocessed-dataset/test.tfrecord'
config.data.validation_ds_path = '/kaggle/input/datasets/migsena/de-en-preprocessed-dataset/validation.tfrecord'
config.data.train_ds_path = '/kaggle/input/datasets/migsena/de-en-preprocessed-dataset/train.tfrecord'

train_ds = load_preprocessed_dataset(config.data.train_ds_path, config.data.max_seq_len)
validation_ds = load_preprocessed_dataset(config.data.validation_ds_path, config.data.max_seq_len)
test_ds = load_preprocessed_dataset(config.data.test_ds_path, config.data.max_seq_len)

In [ ]:
!rm -rf log_dir 
!rm -f log_dir.zip

In [ ]:
config

# Training

In [ ]:
model = create_transformer_module(config)
state = train_and_evaluate(model, 
                           config,
                           main_rng_key,
                           train_ds,
                           validation_ds,
                           log_dir_prefix=None)